# Customer Churn Prediction
Kaggle Playground Series S6E3 — Ensemble of XGBoost + CatBoost + LightGBM with feature engineering.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import lightgbm as lgb

## 2. Load Data

In [2]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")
sub   = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (594194, 21)
Test shape: (254655, 20)


## 3. Feature Engineering

In [3]:
def engineer_features(df):
    df = df.copy()

    # Binary encode Yes/No columns
    yes_no_cols = [
        'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling',
        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
        'TechSupport', 'StreamingTV', 'StreamingMovies'
    ]
    for col in yes_no_cols:
        df[col] = df[col].map({'Yes': 1, 'No': 0, 'No internet service': 0, 'No phone service': 0})

    # Gender
    df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

    # MultipleLines
    df['MultipleLines'] = df['MultipleLines'].map({'Yes': 1, 'No': 0, 'No phone service': 0})

    # InternetService — ordinal: No < DSL < Fiber optic
    df['InternetService'] = df['InternetService'].map({'No': 0, 'DSL': 1, 'Fiber optic': 2})

    # Contract — ordinal: Month-to-month < One year < Two year
    df['Contract'] = df['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year': 2})

    # PaymentMethod — one-hot
    df = pd.get_dummies(df, columns=['PaymentMethod'], drop_first=False)

    # Derived features
    df['ChargesPerMonth']     = df['TotalCharges'] / (df['tenure'] + 1)
    df['ChargesDiff']         = df['MonthlyCharges'] - df['ChargesPerMonth']
    df['TenureGroup']         = pd.cut(df['tenure'], bins=[0,12,24,48,72], labels=[0,1,2,3]).astype(float)
    df['ServiceCount']        = df[['OnlineSecurity','OnlineBackup','DeviceProtection',
                                     'TechSupport','StreamingTV','StreamingMovies']].sum(axis=1)
    df['HasFiberOptic']       = (df['InternetService'] == 2).astype(int)
    df['IsMonthToMonth']      = (df['Contract'] == 0).astype(int)
    df['HighCharges']         = (df['MonthlyCharges'] > df['MonthlyCharges'].median()).astype(int)
    df['LowTenureHighCharge'] = ((df['tenure'] <= 12) & (df['MonthlyCharges'] > 70)).astype(int)
    df['SeniorSingle']        = ((df['SeniorCitizen'] == 1) & (df['Partner'] == 0)).astype(int)

    return df


train_eng = engineer_features(train.drop('Churn', axis=1))
test_eng  = engineer_features(test)

# Align columns
train_eng, test_eng = train_eng.align(test_eng, join='left', axis=1, fill_value=0)

# Target
y = (train['Churn'] == 'Yes').astype(int)

# Drop id
drop_cols = ['id']
X      = train_eng.drop(columns=drop_cols)
X_test = test_eng.drop(columns=drop_cols)

print("Features:", X.shape[1])
X.head(2)

Features: 31


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,PaymentMethod_Mailed check,ChargesPerMonth,ChargesDiff,TenureGroup,ServiceCount,HasFiberOptic,IsMonthToMonth,HighCharges,LowTenureHighCharge,SeniorSingle
0,1,0,1,1,29,1,0,1,1,0,...,True,55.128333,4.971667,2.0,3,0,0,0,0,0
1,1,0,1,1,58,1,0,1,1,1,...,False,64.037288,5.462712,3.0,4,0,0,0,0,0


## 4. Cross-Validation Ensemble

We train XGBoost, LightGBM, and CatBoost inside a 5-fold StratifiedKFold.
Out-of-fold predictions are used for evaluation, and test predictions are averaged.

In [4]:
N_FOLDS = 5
skf     = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_xgb  = np.zeros(len(X))
oof_lgb  = np.zeros(len(X))
oof_cat  = np.zeros(len(X))

test_xgb = np.zeros(len(X_test))
test_lgb = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))

feature_names = X.columns.tolist()

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n--- Fold {fold}/{N_FOLDS} ---")
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    # --- XGBoost ---
    xgb = XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        scale_pos_weight=(y_tr == 0).sum() / (y_tr == 1).sum(),
        use_label_encoder=False,
        eval_metric='auc',
        early_stopping_rounds=50,
        random_state=42,
        tree_method='hist',
        device='cuda',
        verbosity=0
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx]  = xgb.predict_proba(X_val)[:, 1]
    test_xgb         += xgb.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f"  XGB  AUC: {roc_auc_score(y_val, oof_xgb[val_idx]):.5f}")

    # --- LightGBM ---
    lgbm = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=20,
        class_weight='balanced',
        random_state=42,
        verbose=-1
    )
    lgbm.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
    )
    oof_lgb[val_idx]  = lgbm.predict_proba(X_val)[:, 1]
    test_lgb         += lgbm.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f"  LGBM AUC: {roc_auc_score(y_val, oof_lgb[val_idx]):.5f}")

    # --- CatBoost ---
    cat = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=6,
        auto_class_weights='Balanced',
        eval_metric='AUC',
        early_stopping_rounds=50,
        random_seed=42,
        task_type='GPU',
        verbose=0
    )
    cat.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
    oof_cat[val_idx]  = cat.predict_proba(X_val)[:, 1]
    test_cat         += cat.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f"  CAT  AUC: {roc_auc_score(y_val, oof_cat[val_idx]):.5f}")

print("\n=== OOF AUC ===")
print(f"XGBoost  : {roc_auc_score(y, oof_xgb):.5f}")
print(f"LightGBM : {roc_auc_score(y, oof_lgb):.5f}")
print(f"CatBoost : {roc_auc_score(y, oof_cat):.5f}")


--- Fold 1/5 ---
  XGB  AUC: 0.91592
  LGBM AUC: 0.91556


Default metric period is 5 because AUC is/are not implemented for GPU


  CAT  AUC: 0.91532

--- Fold 2/5 ---
  XGB  AUC: 0.91701
  LGBM AUC: 0.91676


Default metric period is 5 because AUC is/are not implemented for GPU


  CAT  AUC: 0.91643

--- Fold 3/5 ---
  XGB  AUC: 0.91645
  LGBM AUC: 0.91605


...

  CAT  AUC: 0.91592

--- Fold 4/5 ---
  XGB  AUC: 0.91755
  LGBM AUC: 0.91736


Default metric period is 5 because AUC is/are not implemented for GPU


  CAT  AUC: 0.91693

--- Fold 5/5 ---
  XGB  AUC: 0.91489
  LGBM AUC: 0.91450


Default metric period is 5 because AUC is/are not implemented for GPU


  CAT  AUC: 0.91434

=== OOF AUC ===
XGBoost  : 0.91635
LightGBM : 0.91604
CatBoost : 0.91578


## 5. Blend & Optimize Weights

In [5]:
from scipy.optimize import minimize

def neg_auc(w):
    w = np.array(w)
    w = np.abs(w) / np.abs(w).sum()
    blend = w[0]*oof_xgb + w[1]*oof_lgb + w[2]*oof_cat
    return -roc_auc_score(y, blend)

res = minimize(neg_auc, x0=[1/3, 1/3, 1/3], method='Nelder-Mead')
w_opt = np.abs(res.x) / np.abs(res.x).sum()
print(f"Optimal weights  XGB={w_opt[0]:.3f}  LGB={w_opt[1]:.3f}  CAT={w_opt[2]:.3f}")

oof_blend  = w_opt[0]*oof_xgb  + w_opt[1]*oof_lgb  + w_opt[2]*oof_cat
test_blend = w_opt[0]*test_xgb + w_opt[1]*test_lgb + w_opt[2]*test_cat
print(f"Blended OOF AUC : {roc_auc_score(y, oof_blend):.5f}")

Optimal weights  XGB=0.495  LGB=0.319  CAT=0.186
Blended OOF AUC : 0.91650


## 6. Submission

In [6]:
sub['Churn'] = test_blend
sub.to_csv('submission.csv', index=False)
print("Saved submission.csv")
sub.head()

Saved submission.csv


,id,Churn
0,594194,0.204852
1,594195,0.001284
2,594196,0.276575
3,594197,0.012599
4,594198,0.778787
